# `quant` — Research Template

End-to-end workflow: **load data → backtest → responsive chart → attribution → sweep → report**.

The charts stay smooth even on **years of 1-minute candles** (millions of points) thanks to
viewport resampling. Run cells top-to-bottom. Tune the config in the second cell.


## 0. Setup
Install once: `pip install -e .` from the repo root (or the cell below adds the repo to the path).


In [ ]:
import sys, os
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..'))   # run from repo root without installing

import warnings; warnings.filterwarnings('ignore')
from quant.data import get_ohlcv
from quant.engine import BacktestConfig, TakeProfit
from quant.strategies import EmaRibbon, RsiReversal, MacdTrend, HeikinAshiTrend, SupertrendFlip, MtfTrend, KeyLevelBounce
from quant.optimize import run_grid
from quant import analytics as A, reporting as R
from quant.viz import ResearchChart, equity_chart, hour_weekday_heatmap, monthly_returns_heatmap, sweep_heatmap
print('quant', quant.__version__, 'ready')

## 1. Load data
Only the requested date range is pulled; missing candles are downloaded incrementally and cached.
Switch `SYMBOL` (e.g. `XAUTUSDT`, `BTCUSDT`) / `TF` (`1m`,`5m`,`15m`,`1h`) / `SOURCE` freely.


In [ ]:
SYMBOL, TF = 'PAXGUSDT', '1m'        # PAX Gold — the cached gold proxy
START, END = '2026-01-01', '2026-05-31'
TZ = 'UTC'                            # display tz for charts & time filters

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source='binance', market='spot', tz=TZ)
print(f'{len(df):,} bars  {df.open_time.min()} -> {df.open_time.max()}')
df.tail(3)

## 2. Configure & run a backtest
`BacktestConfig` is the full execution model: SL / TP / trailing / partials / sizing / fees / slippage.
Every strategy is one dataclass — swap `EmaRibbon(...)` for any other.


In [ ]:
cfg = BacktestConfig(
    initial_cash=10_000, fee_bps=8, slippage_bps=1.5,
    exit_enabled=True,
    sl_mode='entry_pct', sl_value=0.6,          # or 'ref_col' + sl_ref_long_col='swing_last_low'
    tp_mode='rr', tp_value=2.0,                 # or take_profits=(TakeProfit('rr',1,50,'breakeven'), TakeProfit('rr',2.5,100))
    sizing_mode='risk_pct_equity', sizing_value=1.0,
)
strat = EmaRibbon(fast=50, slow=200, confirm_n=5)
res = strat.backtest(df, cfg)
R.print_summary(res, df=df)

## 3. Responsive price chart
Pan/zoom reloads higher resolution for the visible window only — smooth at millions of candles.
Toggle any layer via the legend.


In [ ]:
ch = ResearchChart(df, candles=True)
ch.add_ema(50); ch.add_ema(200)
ch.add_trades(res.trades)
ch.show()

## 4. Equity & drawdown


In [ ]:
equity_chart(res.equity_curve)

## 5. Attribution — when does it work?
Slice performance by session / weekday / hour / market regime.


In [ ]:
A.by_session(res.trades)

In [ ]:
hour_weekday_heatmap(res.trades, metric='total_pnl')

In [ ]:
mr = A.monthly_returns(res.equity_curve)
monthly_returns_heatmap(mr)

In [ ]:
A.by_regime(res.trades, df)   # trend x volatility

## 6. Parameter sweep
Thousands of combos in seconds (bounded parallel; `n_jobs` defaults to CPU-2 to keep the laptop usable).


In [ ]:
grid = {'fast':[20,30,50,75,100], 'slow':[150,200,300], 'confirm_n':[1,3,5,10]}
results = run_grid(df, EmaRibbon, grid, cfg, valid_fn=lambda p: p['fast'] < p['slow'],
                   keep_stats=['num_trades','total_return_pct','sharpe','max_drawdown_pct','profit_factor'])
results.sort_values('sharpe', ascending=False).head(10)

In [ ]:
sweep_heatmap(results, x='fast', y='slow', z='sharpe')

## 7. Standalone HTML report
Offline, shareable — stats + charts + heatmaps in one file.


In [ ]:
path = R.to_html(res, 'reports/research_report.html', df=df, price_df=df, title=f'{SYMBOL} {strat.name}')
print('wrote', path)

## 8. Build your own strategy
A strategy is one dataclass with `prepare()` (add indicator columns) and `build_signals()`
(return entry/exit boolean arrays). Then it backtests and sweeps with zero extra code:

```python
from dataclasses import dataclass
from quant.strategies.base import Strategy
from quant.indicators import add_rsi
from quant.engine import Signals
from quant import signals as S

@dataclass
class MyRsi(Strategy):
    name: str = 'my_rsi'
    period: int = 14
    def prepare(self, df):
        return add_rsi(df, self.period)
    def build_signals(self, df):
        r = f'rsi_{self.period}'
        return Signals(entry_long=S.cross_up(df, r, 30), exit_long=S.cross_down(df, r, 70))

MyRsi(period=14, session='london').backtest(df, cfg).stats   # time filter is free
```
